# 09 - Train-only M0 learning sanity check

Notebook 08 showed small, inconsistent changes, not reliable restoration improvement. Do NOT add a GAN yet. This controlled overfitting diagnostic asks whether the existing architecture can learn fixed synthetic pairs and whether mixing opposite degradations or loss dilution deserves further investigation. These are hypotheses, not established causes.

**T4 is sufficient. Run All.** Requires the frozen artifacts of notebooks 03/05/06/07 and restored train images. No NBIS installation is needed. No validation/test/FVC2004 images are read. The M0 checkpoint is read-only. Diagnostic models must NEVER be substituted for M0 or reported as validation improvements.

Select two eligible train fingers per each of the six source databases, impression 2. Use the existing M0 dataset's fixed 192x192 crop and simulator, with severe dry_like and wet_like conditions. Scan candidates in seed-hash order. Log training-only exclusions when the fixed crop has <1% changed or <1% preserved pixels in either condition. This deliberately informative tiny training set is NOT representative evaluation data. References remain original images, not guaranteed clean.

Four fresh networks, same M0 architecture, initialization seed, Adam lr 0.0002, batch size 8, 300 updates, FP32 without AMP: dry-only global L1; wet-only global L1; mixed global L1; mixed region-balanced L1. The mixed runs use exactly the same patches, initial weights and batch order. No identity samples are included in these optimization runs; identity crops are measured separately to expose preservation damage.

Single-family runs have 12 samples, mixed runs have 24. Equal update budgets do not equalize per-sample exposure. A single/mixed fit gap alone cannot establish gradient conflict or a benefit of separate experts; the matched mixed/global versus mixed/balanced comparison isolates only the loss change in this tiny setting.

Balanced loss = 0.5 * mean absolute error on actually changed pixels + 0.5 * mean error on preserved pixels, calculated per patch then averaged. Changed means abs(synthetic input - reference) > 1e-6; this is synthetic supervision only, NOT an inference-time map or a real moisture/reliability label. This standard diagnostic weighting is not claimed as a novel method.

Select each diagnostic checkpoint by its own objective on this SAME tiny training set, including the initial identity checkpoint. Report all-pixel error, changed/preserved-region errors, quantized changes, per-family errors and original-crop damage. A >=20% reduction in training all-pixel MAE is only a predeclared descriptive sanity threshold, NOT a test accuracy target or proof of generalization. Failure within 300 updates does not prove the architecture is incapable of learning.

The zero-initialized head intentionally blocks encoder gradients on the FIRST update; that alone is not a bug. Log head/body gradients on measured updates and verify that weights actually change. Completed runs are checksummed on Drive. Run All resumes completed runs; an interrupted run is repeated from its deterministic initialization (at most 300 updates). Never rerun only the training cell after an error.

Outputs: MyDrive/FingerprintResearch/diagnostics/m0_learning_sanity_v1/<configuration-hash>/. Completion marker: TRAIN-ONLY LEARNING DIAGNOSTIC COMPLETE. Send the final summary before deciding whether to fix optimization, revise the reconstruction objective, or test a condition-aware model. Held-out tests remain sealed.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
SEED = 20261110
SETTINGS = {'patch_size': 192, 'base_channels': 16, 'batch_size': 8, 'steps': 300,
            'learning_rate': 0.0002, 'log_every': 25, 'per_database': 2,
            'changed_epsilon': 1e-6, 'min_region_fraction': 0.01, 'relative_mae_reduction_gate': 0.20}
RUNS = [('dry_global', 'dry_like', 'global'), ('wet_global', 'wet_like', 'global'),
        ('mixed_global', 'mixed', 'global'), ('mixed_balanced', 'mixed', 'balanced')]

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
print('GPU:', torch.cuda.get_device_name(0), '| Four small FP32 learning checks; no validation or matching.')

In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
for name, expected in split_protocol['file_sha256'].items():
    if file_digest(SPLIT_DIR / name) != expected:
        raise RuntimeError(f'Frozen split changed: {name}')
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
train_config = json.loads((EXPERIMENT_DIR / 'configuration.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
best = json.loads((EXPERIMENT_DIR / 'best.json').read_text(encoding='utf-8'))
train_completion = json.loads((EXPERIMENT_DIR / 'completion.json').read_text(encoding='utf-8'))
train_sha = file_digest(EXPERIMENT_DIR / 'configuration.json')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
       for config in (train_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (train_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or train_config['simulator_module_sha256'] != sim_config['module_sha256']
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']):
    raise RuntimeError('Simulator/normalization provenance mismatch.')
for name, expected in train_config['source_file_sha256'].items():
    if file_digest(EXPERIMENT_DIR / name) != expected:
        raise RuntimeError(f'Frozen training source changed: {name}')
if (train_completion['status'] != 'm0_training_complete'
        or train_completion['epochs_completed'] != train_config['epochs']
        or train_completion['configuration_sha256'] != train_sha
        or train_completion['best_checkpoint'] != best or best['configuration_sha256'] != train_sha
        or file_digest(EXPERIMENT_DIR / best['file']) != best['sha256']):
    raise RuntimeError('M0 completion/checkpoint provenance mismatch.')
if (str(torch.__version__) != train_config['torch_version'] or str(torch.version.cuda) != train_config['torch_cuda']
        or Image.__version__ != train_config['pillow_version'] or Image.__version__ != pair_config['pillow_version']
        or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']):
    raise RuntimeError('Runtime versions differ from the frozen training experiment.')
if train_config['base_channels'] != SETTINGS['base_channels'] or train_config['patch_size'] != SETTINGS['patch_size']:
    raise RuntimeError('Diagnostic architecture/patch size must match M0.')
training = load_module('m0_diagnostic_model_v1', EXPERIMENT_DIR / 'restoration_baseline.py')
simulator = load_module('m0_diagnostic_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
candidates = pd.read_csv(EXPERIMENT_DIR / 'train_sources.csv', keep_default_na=False)
allowed = locked[locked['split'].eq('train')].set_index('relative_path')
if candidates['relative_path'].duplicated().any():
    raise RuntimeError('Duplicate train-source paths.')
for row in candidates.to_dict('records'):
    path = row['relative_path']
    if (path not in allowed.index or allowed.loc[path, 'sha256'] != row['sha256']
            or allowed.loc[path, 'identity_key'] != row['identity_key']
            or row['dataset'] not in ('FVC2000', 'FVC2002')):
        raise RuntimeError(f'Non-train/mismatched diagnostic candidate: {path}')
candidates = candidates[candidates['impression_id'].eq(2)].copy()
candidates['selection_key'] = candidates['relative_path'].map(
    lambda path: hashlib.sha256(f'{SEED}|learning_diagnostic|{path}'.encode()).hexdigest())
reference_model = training.RestorationUNet(base=SETTINGS['base_channels']).to(DEVICE)
checkpoint = torch.load(EXPERIMENT_DIR / best['file'], map_location='cpu', weights_only=True)
if checkpoint['configuration_sha256'] != train_sha or checkpoint['epoch'] != best['epoch']:
    raise RuntimeError('M0 checkpoint payload mismatch.')
reference_model.load_state_dict(checkpoint['model'], strict=True)
reference_model.eval()
del checkpoint
print('Verified train-only candidates:', len(candidates), '| M0 epoch:', best['epoch'] + 1)

In [ ]:
selected_records, sample_rows, selection_rows, identity_rows = [], [], [], []
for (dataset, db), group in candidates.groupby(['dataset', 'db'], sort=True):
    accepted = 0
    for record in group.sort_values('selection_key').drop(columns='selection_key').to_dict('records'):
        path = DATA_ROOT / record['relative_path']
        if not path.is_file() or file_digest(path) != record['sha256']:
            raise RuntimeError(f'Train source missing/changed: {path}. Restore images with notebook 02.')
        dataset_object = training.FingerprintPairs([record], DATA_ROOT, simulator, pair_config['source_dpi'],
            SEED, patch_size=SETTINGS['patch_size'], training=False)
        samples = []
        for family in ('dry_like', 'wet_like'):
            index = dataset_object.conditions.index((family, 'severe'))
            sample = dataset_object[index]
            changed = (sample['input'] - sample['target']).abs().gt(SETTINGS['changed_epsilon'])
            fraction = float(changed.float().mean())
            samples.append({**sample, 'family': family, 'changed': changed, 'changed_fraction': fraction})
        eligible = all(SETTINGS['min_region_fraction'] <= sample['changed_fraction'] <= 1 - SETTINGS['min_region_fraction']
                       for sample in samples)
        selection_rows.append({'relative_path': record['relative_path'], 'dataset': dataset, 'db': db,
            'decision': 'included' if eligible else 'excluded',
            'reason': 'informative_fixed_training_crop' if eligible else 'insufficient_changed_or_preserved_crop_area',
            'dry_changed_fraction': samples[0]['changed_fraction'], 'wet_changed_fraction': samples[1]['changed_fraction']})
        if not eligible:
            continue
        if not torch.equal(samples[0]['target'], samples[1]['target']):
            raise RuntimeError('Dry/wet diagnostic conditions do not share the same target crop.')
        selected_records.append(record)
        sample_rows.extend(samples)
        identity_rows.append(samples[0]['target'])
        accepted += 1
        if accepted == SETTINGS['per_database']:
            break
    if accepted != SETTINGS['per_database']:
        raise RuntimeError(f'Too few informative train crops in {dataset}/{db}; inspect the selection log, do not use validation.')
if len(selected_records) != 6 * SETTINGS['per_database']:
    raise RuntimeError('Expected train-only selection from all six databases.')
if len({row['identity_key'] for row in selected_records}) != len(selected_records):
    raise RuntimeError('Diagnostic sources must be different scoped train fingers.')
inputs = torch.stack([row['input'] for row in sample_rows])
targets = torch.stack([row['target'] for row in sample_rows])
changed = torch.stack([row['changed'] for row in sample_rows])
identity_inputs = torch.stack(identity_rows)
families = [row['family'] for row in sample_rows]
sample_audit = pd.DataFrame([{name: row[name] for name in ('relative_path', 'family', 'changed_fraction')}
                             for row in sample_rows])
print('Fixed training patches:', len(inputs), '| Original reference crops:', len(identity_inputs))
print(pd.DataFrame(selection_rows).to_string(index=False))
print('No validation or test image has been opened.')

In [ ]:
def new_model():
    torch.manual_seed(SEED)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
    return training.RestorationUNet(base=SETTINGS['base_channels']).to(DEVICE)

def objective_loss(predictions, reference, support, objective):
    error = (predictions - reference).abs().flatten(1)
    if objective == 'global':
        return error.mean()
    if objective != 'balanced':
        raise ValueError(f'Unknown objective: {objective}')
    mask = support.flatten(1).to(error.dtype)
    active, preserved = mask.sum(1), (1 - mask).sum(1)
    if (active == 0).any() or (preserved == 0).any():
        raise ValueError('Balanced diagnostic requires changed AND preserved pixels in every training patch.')
    return (0.5 * (error * mask).sum(1) / active + 0.5 * (error * (1 - mask)).sum(1) / preserved).mean()

@torch.inference_mode()
def measure(model, xs, ys, support):
    if len(xs) == 0 or len(xs) != len(ys) or xs.shape != ys.shape or support.shape != xs.shape:
        raise ValueError('Expected nonempty aligned diagnostic tensors.')
    model.eval()
    prediction_rows = []
    for offset in range(0, len(xs), SETTINGS['batch_size']):
        output = model(xs[offset:offset + SETTINGS['batch_size']].to(DEVICE)).float().cpu()
        if not torch.isfinite(output).all() or output.min() < 0 or output.max() > 1:
            raise RuntimeError('Non-finite/out-of-range diagnostic output.')
        prediction_rows.append(output)
    predictions = torch.cat(prediction_rows)
    error = (predictions - ys).abs().flatten(1)
    mask = support.flatten(1).float()
    active, preserved = mask.sum(1), (1 - mask).sum(1)
    has_active, has_preserved = active > 0, preserved > 0
    input_mae = float((xs - ys).abs().mean())
    output_mae = float(error.mean())
    before_png, after_png = torch.round(xs * 255), torch.round(predictions * 255)
    return {'samples': len(xs), 'input_mae': input_mae, 'output_mae': output_mae,
        'relative_mae_reduction': (input_mae - output_mae) / input_mae if input_mae > 0 else None,
        'changed_region_mae': float(((error * mask).sum(1)[has_active] / active[has_active]).mean()) if has_active.any() else None,
        'preserved_region_mae': float(((error * (1 - mask)).sum(1)[has_preserved] / preserved[has_preserved]).mean()) if has_preserved.any() else None,
        'changed_region_samples': int(has_active.sum()), 'preserved_region_samples': int(has_preserved.sum()),
        'png_changed_pixel_fraction': float(before_png.ne(after_png).float().mean()),
        'png_identical_image_fraction': float(before_png.eq(after_png).flatten(1).all(1).float().mean())}

def tensor_digest(tensor):
    array = tensor.contiguous().cpu().numpy()
    return hashlib.sha256(json_bytes({'shape': list(array.shape), 'dtype': str(array.dtype)}) + array.tobytes()).hexdigest()

def load_run(directory):
    marker = directory / 'completion.json'
    if not marker.is_file():
        return None
    saved = json.loads(marker.read_text(encoding='utf-8'))
    expected_files = {'history.csv', 'results.json', saved['weights_file']}
    if (saved['configuration_sha256'] != diagnostic_sha or saved['run'] != directory.name
            or set(saved['file_sha256']) != expected_files):
        raise RuntimeError(f'Completed diagnostic configuration/files mismatch: {directory}')
    for name, expected in saved['file_sha256'].items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Completed diagnostic integrity failure: {directory / name}')
    return json.loads((directory / 'results.json').read_text(encoding='utf-8'))

def fit_run(name, family, objective):
    directory = REPORT_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    resumed = load_run(directory)
    if resumed is not None:
        print('Resumed completed diagnostic:', name, flush=True)
        return resumed
    selected = [index for index, value in enumerate(families) if family == 'mixed' or value == family]
    xs, ys, support = inputs[selected], targets[selected], changed[selected]
    model = new_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=SETTINGS['learning_rate'])
    initial = measure(model, xs, ys, support)
    if abs(initial['output_mae'] - initial['input_mae']) > 1e-7:
        raise RuntimeError('Fresh network must start as the identity mapping.')
    initial_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    best_state = initial_state
    best_step = 0
    best_objective = initial['output_mae'] if objective == 'global' else 0.5 * (initial['changed_region_mae'] + initial['preserved_region_mae'])
    history = [{'step': 0, 'objective': best_objective, 'train_global_mae': initial['output_mae'],
                'head_gradient_l2': None, 'body_gradient_l2': None}]
    seen = set()
    for step in range(1, SETTINGS['steps'] + 1):
        rng = np.random.default_rng(training.stable_seed(f'{SEED}|diagnostic_batch|{step}'))
        batch = rng.choice(len(xs), size=min(SETTINGS['batch_size'], len(xs)), replace=False).tolist()
        seen.update(batch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        prediction = model(xs[batch].to(DEVICE))
        loss = objective_loss(prediction, ys[batch].to(DEVICE), support[batch].to(DEVICE), objective)
        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite loss: {name}, step {step}')
        loss.backward()
        log_step = step == 1 or step % SETTINGS['log_every'] == 0 or step == SETTINGS['steps']
        head_norm, body_norm = None, None
        if log_step:
            norms = {'head': [], 'body': []}
            for key, parameter in model.named_parameters():
                if parameter.grad is not None:
                    norms['head' if key.startswith('head.') else 'body'].append(parameter.grad.detach().square().sum())
            head_norm = float(torch.stack(norms['head']).sum().sqrt())
            body_norm = float(torch.stack(norms['body']).sum().sqrt())
            if not math.isfinite(head_norm) or not math.isfinite(body_norm):
                raise RuntimeError(f'Non-finite gradients: {name}, step {step}')
        optimizer.step()
        if log_step:
            current = measure(model, xs, ys, support)
            current_objective = current['output_mae'] if objective == 'global' else 0.5 * (current['changed_region_mae'] + current['preserved_region_mae'])
            history.append({'step': step, 'objective': current_objective, 'train_global_mae': current['output_mae'],
                            'head_gradient_l2': head_norm, 'body_gradient_l2': body_norm})
            if current_objective < best_objective:
                best_objective, best_step = current_objective, step
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            print(f'{name}: step {step}/{SETTINGS["steps"]}, objective={current_objective:.5f}, global MAE={current["output_mae"]:.5f}', flush=True)
    if len(seen) != len(xs):
        raise RuntimeError(f'Not every selected training patch was sampled: {name}')
    last_head_change = float((model.head.weight.detach().cpu() - initial_state['head.weight']).norm())
    last_encoder_change = float((model.enc1.layers[0].weight.detach().cpu() - initial_state['enc1.layers.0.weight']).norm())
    model.load_state_dict(best_state)
    trained = measure(model, xs, ys, support)
    evaluations = {}
    for condition in ('dry_like', 'wet_like'):
        indices = [index for index, value in enumerate(families) if value == condition]
        evaluations[condition] = measure(model, inputs[indices], targets[indices], changed[indices])
    evaluations['identity'] = measure(model, identity_inputs, identity_inputs, torch.zeros_like(identity_inputs, dtype=torch.bool))
    relative_drop = trained['relative_mae_reduction']
    result = {'run': name, 'family': family, 'objective': objective, 'updates': SETTINGS['steps'],
        'best_step': best_step, 'best_objective': best_objective, 'initial_training': initial,
        'best_training': trained, 'training_mae_reduction_gate_met': relative_drop >= SETTINGS['relative_mae_reduction_gate'],
        'best_head_weight_change_l2': float((best_state['head.weight'] - initial_state['head.weight']).norm()),
        'best_encoder_weight_change_l2': float((best_state['enc1.layers.0.weight'] - initial_state['enc1.layers.0.weight']).norm()),
        'last_head_weight_change_l2': last_head_change, 'last_encoder_weight_change_l2': last_encoder_change,
        'fixed_train_evaluations': evaluations, 'generalization_evaluated': False}
    local_weights = LOCAL_DIAGNOSTIC / f'{name}.pt'
    torch.save({'configuration_sha256': diagnostic_sha, 'run': name, 'best_step': best_step,
                'model': best_state}, local_weights)
    weights_sha = file_digest(local_weights)
    weights_name = f'model_{weights_sha[:16]}.pt'
    destination = directory / weights_name
    pending = destination.with_suffix('.pt.part')
    shutil.copyfile(local_weights, pending)
    if file_digest(pending) != weights_sha:
        raise RuntimeError(f'Diagnostic checkpoint Drive copy failed: {name}')
    pending.replace(destination)
    local_weights.unlink()
    write_csv(directory / 'history.csv', pd.DataFrame(history))
    write_verified(directory / 'results.json', json_bytes(result))
    marker = {'configuration_sha256': diagnostic_sha, 'run': name, 'weights_file': weights_name,
              'file_sha256': {filename: file_digest(directory / filename)
                             for filename in ('history.csv', 'results.json', weights_name)}}
    write_verified(directory / 'completion.json', json_bytes(marker))
    return result

In [ ]:
helper_source = '\n\n'.join(inspect.getsource(function)
    for function in (new_model, objective_loss, measure, tensor_digest, load_run, fit_run))
source_files = {
    'selected_train_sources.csv': pd.DataFrame(selected_records).to_csv(index=False, lineterminator='\n').encode(),
    'selection_log.csv': pd.DataFrame(selection_rows).to_csv(index=False, lineterminator='\n').encode(),
    'patch_audit.csv': sample_audit.to_csv(index=False, lineterminator='\n').encode(),
    'diagnostic_helpers.py.txt': helper_source.encode('utf-8'),
}
diagnostic_config = {'version': 'm0_learning_sanity_v1', 'seed': SEED, 'settings': SETTINGS, 'runs': RUNS,
    'training_configuration_sha256': train_sha, 'm0_best_checkpoint': best,
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'selection_policy': 'two informative fixed train crops per source database; impression 2; severe dry/wet',
    'batch_policy': 'without replacement within each update; per-step seed; same order for both mixed objectives',
    'checkpoint_selection': 'lowest own training objective among logged updates, including initial identity',
    'tensor_sha256': {name: tensor_digest(value) for name, value in
                      (('inputs', inputs), ('targets', targets), ('changed', changed), ('identity', identity_inputs))},
    'file_sha256': {name: hashlib.sha256(data).hexdigest() for name, data in source_files.items()},
    'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
    'pillow_version': Image.__version__, 'numpy_version': np.__version__, 'precision': 'FP32; AMP/TF32 disabled',
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'gpu_compute_capability': list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None,
    'validation_or_test_evaluated': False,
}
config_data = json_bytes(diagnostic_config)
diagnostic_sha = hashlib.sha256(config_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'diagnostics' / 'm0_learning_sanity_v1' / diagnostic_sha[:20]
LOCAL_DIAGNOSTIC = LOCAL_ROOT / 'diagnostics' / diagnostic_sha[:20]
for directory in (REPORT_DIR, LOCAL_DIAGNOSTIC):
    directory.mkdir(parents=True, exist_ok=True)
for name, data in {**source_files, 'configuration.json': config_data}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen diagnostic changed: {path}')
    if not path.exists():
        write_verified(path, data)
_diagnostic_training_entered = False
print('Train-only diagnostic frozen:', diagnostic_sha)
print('Drive outputs:', REPORT_DIR)

In [ ]:
if _diagnostic_training_entered:
    raise RuntimeError('Use Run All to resume safely; do not rerun only the training cell.')
_diagnostic_training_entered = True
results = [fit_run(name, family, objective) for name, family, objective in RUNS]
print('All four train-only diagnostics completed and saved.')

In [ ]:
summary_rows = []
for result in results:
    summary_rows.append({'run': result['run'], 'best_step': result['best_step'],
        'training_patches': result['best_training']['samples'],
        'input_train_mae': result['initial_training']['input_mae'],
        'best_train_mae': result['best_training']['output_mae'],
        'relative_train_mae_reduction': result['best_training']['relative_mae_reduction'],
        'changed_region_mae': result['best_training']['changed_region_mae'],
        'preserved_region_mae': result['best_training']['preserved_region_mae'],
        'identity_crop_mae': result['fixed_train_evaluations']['identity']['output_mae'],
        'training_reduction_gate_met': result['training_mae_reduction_gate_met'],
        'best_encoder_weight_change_l2': result['best_encoder_weight_change_l2'],
        'last_head_weight_change_l2': result['last_head_weight_change_l2'],
        'last_encoder_weight_change_l2': result['last_encoder_weight_change_l2']})
summary = pd.DataFrame(summary_rows)
write_csv(REPORT_DIR / 'learning_summary.csv', summary)
baseline_rows = []
for family in ('dry_like', 'wet_like'):
    indices = [index for index, value in enumerate(families) if value == family]
    baseline_rows.append({'family': family, **measure(reference_model, inputs[indices], targets[indices], changed[indices])})
baseline_rows.append({'family': 'identity', **measure(reference_model, identity_inputs, identity_inputs,
                                                    torch.zeros_like(identity_inputs, dtype=torch.bool))})
write_csv(REPORT_DIR / 'frozen_m0_train_crop_metrics.csv', pd.DataFrame(baseline_rows))
print(summary.to_string(index=False))
print('\nFrozen M0 on the SAME fixed training crops:')
print(pd.DataFrame(baseline_rows).to_string(index=False))

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, family, objective in RUNS:
    history = pd.read_csv(REPORT_DIR / name / 'history.csv')
    axes[0].plot(history['step'], history['train_global_mae'], label=name)
    axes[1].plot(history['step'], history['body_gradient_l2'], label=name)
axes[0].set_title('All-pixel MAE on each run training set')
axes[1].set_title('Body gradient L2 on logged minibatches')
for axis in axes:
    axis.set_xlabel('Optimizer update')
    axis.legend(fontsize=7)
figure.tight_layout()
figure.savefig(REPORT_DIR / 'learning_curves.png', dpi=140)
plt.show()

models = {'M0': reference_model}
for name, family, objective in RUNS:
    marker = json.loads((REPORT_DIR / name / 'completion.json').read_text(encoding='utf-8'))
    model = new_model()
    state = torch.load(REPORT_DIR / name / marker['weights_file'], map_location='cpu', weights_only=True)
    if state['configuration_sha256'] != diagnostic_sha or state['run'] != name:
        raise RuntimeError('Diagnostic preview checkpoint payload mismatch.')
    model.load_state_dict(state['model'], strict=True)
    model.eval()
    models[name] = model
columns = ['input', 'M0'] + [name for name, _, _ in RUNS] + ['reference']
figure, axes = plt.subplots(2, len(columns), figsize=(17, 5))
with torch.inference_mode():
    for row in range(2):
        for column, name in enumerate(columns):
            if name == 'input':
                image = inputs[row, 0].numpy()
            elif name == 'reference':
                image = targets[row, 0].numpy()
            else:
                image = models[name](inputs[row:row + 1].to(DEVICE)).cpu()[0, 0].numpy()
            axes[row, column].imshow(np.rint(image * 255).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
            axes[row, column].set_title(f'{families[row]} | {name}', fontsize=7)
            axes[row, column].axis('off')
figure.suptitle('Fixed first train source only; memorization check, NOT validation')
figure.tight_layout()
figure.savefig(REPORT_DIR / 'training_patch_preview.png', dpi=140)
plt.show()
files = ['configuration.json', *source_files, 'learning_summary.csv', 'frozen_m0_train_crop_metrics.csv',
         'learning_curves.png', 'training_patch_preview.png']
completion = {'status': 'train_only_learning_diagnostic_complete', 'configuration_sha256': diagnostic_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in files},
    'run_completion_sha256': {name: file_digest(REPORT_DIR / name / 'completion.json') for name, _, _ in RUNS},
    'validation_or_test_evaluated': False, 'recognition_improvement_verified': False,
    'diagnostic_models_are_not_research_baselines': True}
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
print('TRAIN-ONLY LEARNING DIAGNOSTIC COMPLETE:', REPORT_DIR)
print('Send learning_summary and learning_curves. A training-fit gain is NOT a recognition/generalization result.')